# 09_phase3_PAGA_trajectory.ipynb
Phase 3 stretch — PAGA trajectory inference

**Scope:** GSE114725 T cell sub-clusters (CD4 Naive/Resting, CD4 Activated, Regulatory T cells, NK/Cytotoxic T cells) — chosen because trajectory inference is most meaningful within a biologically continuous population, and this sub-cluster set was already validated in Phase 2.

**What PAGA does:** builds a graph of connectivity between clusters (not just nearest-neighbour distances within a cluster) — showing which cluster transitions are well-supported by intermediate cells vs which clusters are transcriptionally disconnected. This is a coarse, cluster-level trajectory estimate, not a full pseudotime ordering of individual cells (that would be a natural follow-up if this shows a clear structure worth pursuing further).

In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sc.settings.verbosity = 1

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_paga"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_paga"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [6]:
import os
path = PROCESSED_DIR / "GSE114725_tcells_subclustered.h5ad"
print(f"File modified: {os.path.getmtime(path)}")
import datetime
print(datetime.datetime.fromtimestamp(os.path.getmtime(path)))

print("\nActual categories in this file's tcell_subtype column:")
print(adata_tcells.obs["tcell_subtype"].value_counts(dropna=False))

File modified: 1783699888.3920436
2026-07-10 17:11:28.392044

Actual categories in this file's tcell_subtype column:
tcell_subtype
nan                     11784
Resting T cells          6681
Naive/Memory T cells     6121
Mast cells               3943
NK/Cytotoxic T cells     2529
Activated T cells        1945
Name: count, dtype: int64


In [2]:
# ----------------------------
# Cell 2 — Load T cell sub-clustered data
# ----------------------------
adata_tcells = sc.read_h5ad(PROCESSED_DIR / "GSE114725_tcells_subclustered.h5ad")
print(f"Loaded: {adata_tcells.n_obs} cells, {adata_tcells.n_vars} genes")
print(adata_tcells.obs["tcell_subtype"].value_counts())

Loaded: 33003 cells, 2000 genes
tcell_subtype
Resting T cells         6681
Naive/Memory T cells    6121
Mast cells              3943
NK/Cytotoxic T cells    2529
Activated T cells       1945
Name: count, dtype: int64


In [4]:
adata_tcells.obs["tcell_subtype"] = adata_tcells.obs["tcell_subtype"].astype(str).astype("category")

In [5]:
# ----------------------------
# Cell 3 — Run PAGA
# Uses the existing neighbour graph from sub-clustering (already computed
# on X_pca_harmony, seeded, single-threaded — same reproducibility
# standard as the rest of the pipeline). PAGA itself is deterministic
# given a fixed neighbour graph, no additional seeding needed.
# ----------------------------
sc.tl.paga(adata_tcells, groups="tcell_subtype")

print("PAGA connectivity matrix (edge weight = confidence of connection):")
paga_connectivities = pd.DataFrame(
    adata_tcells.uns["paga"]["connectivities"].toarray(),
    index=adata_tcells.obs["tcell_subtype"].cat.categories,
    columns=adata_tcells.obs["tcell_subtype"].cat.categories,
)
print(paga_connectivities.round(3))
paga_connectivities.to_csv(RESULTS_DIR / "GSE114725_tcell_paga_connectivities.csv")

PAGA connectivity matrix (edge weight = confidence of connection):
                      Activated T cells  Mast cells  NK/Cytotoxic T cells  \
Activated T cells                 0.000       0.216                 0.000   
Mast cells                        0.216       0.000                 0.002   
NK/Cytotoxic T cells              0.000       0.002                 0.000   
Naive/Memory T cells              0.249       0.293                 0.000   
Resting T cells                   0.250       0.740                 0.000   
nan                               0.075       0.053                 0.168   

                      Naive/Memory T cells  Resting T cells    nan  
Activated T cells                    0.249            0.250  0.075  
Mast cells                           0.293            0.740  0.053  
NK/Cytotoxic T cells                 0.000            0.000  0.168  
Naive/Memory T cells                 0.000            0.188  0.183  
Resting T cells                      0.188      

In [ ]:
# ----------------------------
# Cell 4 — PAGA graph visualisation
# Line thickness = connectivity strength between cluster pairs.
# Thick lines = well-supported transitions (many intermediate cells);
# thin/absent lines = transcriptionally distinct, no clear intermediate
# population connecting them.
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.paga(adata_tcells, color="tcell_subtype", threshold=0.05,
           node_size_scale=2, edge_width_scale=1.5,
           fontsize=10, frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_tcell_paga_graph.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("PAGA graph saved")

In [ ]:
# ----------------------------
# Cell 5 — PAGA-initialised UMAP
# Standard practice: use PAGA's coarse structure to initialise UMAP
# layout, producing a more trajectory-faithful embedding than a
# default random initialisation.
# ----------------------------
sc.tl.umap(adata_tcells, init_pos="paga", random_state=42)

fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.umap(adata_tcells, color="tcell_subtype",
           title="GSE114725 T cells — PAGA-initialised UMAP",
           legend_loc="right margin", legend_fontsize=9,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_tcell_paga_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

adata_tcells.write(PROCESSED_DIR / "GSE114725_tcells_paga.h5ad", compression="gzip")
print("PAGA-initialised UMAP saved, object saved with PAGA results embedded")

In [ ]:
# ----------------------------
# Cell 6 — Interpretation summary
# ----------------------------
print("=== PAGA connectivity summary ===")
print("Highest-connectivity pairs (excluding self-connections):\n")

conn = paga_connectivities.copy()
np.fill_diagonal(conn.values, 0)
pairs = []
for i, row in enumerate(conn.index):
    for j, col in enumerate(conn.columns):
        if j > i:
            pairs.append((row, col, conn.iloc[i, j]))
pairs_df = pd.DataFrame(pairs, columns=["Cluster A", "Cluster B", "Connectivity"])
pairs_df = pairs_df.sort_values("Connectivity", ascending=False)
print(pairs_df.to_string(index=False))
pairs_df.to_csv(RESULTS_DIR / "GSE114725_tcell_paga_pairwise_connectivity.csv", index=False)

print("\n>>> Interpretation guide: high connectivity between two clusters suggests")
print(">>> a plausible transition/continuum between them (e.g. Naive -> Activated")
print(">>> would be biologically expected to connect). Low/zero connectivity between")
print(">>> two clusters suggests they are transcriptionally distinct end-states,")
print(">>> not part of the same differentiation path.")